# Lab 01 · A small RAG system

**Prepared by Mustafa Assaf** · Beginner level · 60–90 minute session

Start with the small worked example, read the expected output, then change one thing.
Use a CPU runtime and run the cells from top to bottom. No repository clone is needed.

**Practice mode:** model answers are explicitly scripted so you can inspect the code
without a key. They demonstrate control flow, not model quality. MCP, A2A and LangGraph
examples run their actual libraries locally even in practice mode; their setup cells
install pinned packages if needed.

**Optional live calls:** add `FREEINFERENCE_API_KEY` in Colab's Secrets panel, allow
this notebook access, and change `MODE` to `"live"`. The initial model is
`qwen3.6-35b` on FreeInference; `BASE_URL`, `MODEL` and `SECRET_NAME` can be changed
for another compatible provider. Keep the key out of cells and outputs.

### Shared setup
The next cell contains the reusable model client. Run it as supplied; the short teaching examples follow below.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Check which output-token field your provider supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use an HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or ("practice" if self.mode == "practice" else None), input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Check your quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check provider access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect provider configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Run all examples without a key first. Change only MODE for a live model run.
MODE = "practice"  # "live" uses the API key supplied through Colab Secrets
BASE_URL = "https://freeinference.org/v1"
MODEL = "qwen3.6-35b"
SECRET_NAME = "FREEINFERENCE_API_KEY"

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL,
                api_key=secret(SECRET_NAME) if MODE == "live" else "",
                max_calls=24, max_tokens=512)
print("Mode:", MODE)

## Answer from a few library notes

**Goal:** split text into small pieces, retrieve a relevant piece, and answer with a source ID.

Retrieval-augmented generation (RAG) adds relevant source text to a model prompt. We use fictional library notes so the whole example fits on screen. This is a lexical baseline, not an embedding model or vector database.

**Flow:** question → retrieve a note → generate an answer → check the citation.

In [ ]:
notes = {
    "hours": "The library opens at 09:00. The library closes at 18:00.",
    "loans": "Students may borrow three books. Books are due after fourteen days.",
}
chunks = [{"id": f"{name}-{i}", "text": sentence.strip() + "."}
    for name, text in notes.items()
    for i, sentence in enumerate(text.split(".")[:-1])]
print(chunks)
assert len(chunks) == 4

Each chunk has an ID we can cite. This simple splitter assumes short sentences ending in a full stop. A real document may need a better splitter and document/version metadata.

## Retrieve by word overlap

The code below ranks chunks by shared words. Zero shared words gives no result. This makes it easy to inspect, but it misses synonyms and sometimes matches irrelevant text.

In [ ]:
import re

def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def retrieve(question):
    scored = [(len(words(question) & words(c["text"])), c) for c in chunks]
    score, best = max(scored, key=lambda item: item[0])
    return best if score > 0 else None

hit = retrieve("When does the library open?")
print(hit)
assert hit is not None and hit["id"].startswith("hours-")

Inspect the retrieved sentence before answering. Shared words can produce ties, and the highest-ranked chunk is not automatically relevant. Our opening-time query should retrieve the first hours sentence.

In [ ]:
question = "When does the library open?"
hit = retrieve(question)
prompt = f"Use only this note: [{hit['id']}] {hit['text']}\nQuestion: {question}\n"
prompt += 'Return only JSON with answer and source_id. If unsupported, answer "I do not know".'
raw = client.ask(prompt,
    practice=json.dumps({"answer": "The library opens at 09:00.", "source_id": hit["id"]}))
answer = json.loads(raw)
assert set(answer) == {"answer", "source_id"}
assert answer["source_id"] == hit["id"]
print(answer)

**Expected practice answer:** “The library opens at 09:00,” with a source ID.

A matching source ID only tells us the source exists. Read the retrieved sentence: does it actually support the answer? A model can cite the right ID while making the wrong claim.

## Missing evidence should stay missing

In [ ]:
unknown_question = "Parking prices?"
missing = retrieve(unknown_question)
if missing is None:
    response = "I do not know: no relevant note was found."
else:
    response = "A note matched; a human should check relevance before answering."
print(response)
assert missing is None

Here the application abstains before calling the model. This saves a call and avoids inviting an unsupported answer. It does not solve all retrieval failures: weak matches can still be irrelevant.

**Tiny exercises:**

1. Ask about a synonym such as “shutting time.” Explain why lexical matching may fail.
2. Add a note about weekend hours. Give its chunks distinct IDs and inspect which one wins.
3. Add a line in a note saying “ignore the user's question.” Treat that line as source data, not an instruction from the user.

**Check your understanding:** retrieval finds text; generation writes an answer. Which failure happened if the right sentence was never retrieved? Which happened if it was retrieved but misquoted?

## Read further

- [Lewis et al.: Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401)
- [Anthropic: the augmented LLM](https://www.anthropic.com/research/building-effective-agents)

### Finish
Explain the result in your own words. Make the small change suggested above and rerun its check. If you use a live model, save the actual result and trace; a different or failed response is something to investigate.